# Challenge de la Sesión 01 — Temperatura, diversidad y decisión

**Introducción a modelos generativos y preentrenados** · *Desarrollo y Evaluación de Modelos Propios* · Universidad del Rosario · Rosario GSB

[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/juccaicedoac03/Desarrollo-Evaluacion-Modelos/blob/main/sessions/01-intro-generative-models/challenge.ipynb)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/juccaicedoac03/Desarrollo-Evaluacion-Modelos/blob/main/sessions/01-intro-generative-models/challenge.ipynb)

| | |
|---|---|
| **Modalidad** | Individual |
| **Tiempo** | 40 minutos en clase · entrega hasta las **23:59 de hoy** (hora de Colombia) |
| **Uso de IA** | Permitido, con **registro obligatorio** al final del notebook ([política de uso de IA](https://github.com/juccaicedoac03/Desarrollo-Evaluacion-Modelos/blob/main/docs/politica-uso-ia.md)) |
| **Micro-sustentaciones** | Durante el challenge se eligen 3–4 estudiantes al azar para explicar su notebook en 2–3 minutos, sin IA. Si no puedes explicar tu entrega, la nota máxima de este challenge es **3.0** |
| **Rúbrica** | Ejecución técnica **40%** · Análisis e interpretación **40%** · Reflexión y registro de IA **20%** ([evaluación](https://github.com/juccaicedoac03/Desarrollo-Evaluacion-Modelos/blob/main/docs/evaluacion.md)) |
| **Entrega** | `Archivo → Descargar notebook` (*File → Download notebook*) y súbelo a **e-Aulas** como **`S01_<codigo>.ipynb`** (por ejemplo, `S01_123456.ipynb`) |

### Cómo trabajar
1. En Kaggle, activa **Settings → Internet → On**. La CPU es suficiente; la GPU es opcional.
2. Escribe tu **código estudiantil** y tu nombre en la primera celda de código y ejecuta el notebook **en orden** (*Run All*).
   Tu configuración personal (prompts, temperaturas y escenario) depende de tu código: tus números serán distintos de los de tus compañeros.
3. Completa las celdas marcadas con ✏️ y responde en las celdas que empiezan con **Tu respuesta:**, citando **tus** números.
4. Antes de entregar, verifica que aparezcan **"Tu configuración personal"** y la **huella de resultados** al final.

| Tarea | Qué haces | Componente de la rúbrica |
|---|---|---|
| 1 · Ejecución | Generas 5 muestras por prompt × temperatura y mides su diversidad | Ejecución técnica (40%) |
| 2 · Análisis | Interpretas cómo la temperatura cambió diversidad y calidad; eliges la mejor temperatura por prompt | Análisis e interpretación (40%) |
| 3 · Explica y decide | Ubicas tu escenario de negocio en el espectro de adaptación | Análisis e interpretación (40%) |
| 4 · Crítica a la IA | Encuentras, explicas y corriges 3 errores en un texto sobre BERT y GPT | Análisis e interpretación (40%) |
| Reflexión y registro de IA | Al final del notebook | Reflexión y registro de IA (20%) |

In [ ]:
# ✏️ Escribe tu código estudiantil (como aparece en e-Aulas) y tu nombre completo.
STUDENT_ID = ""
STUDENT_NAME = ""

# --- Configuración del curso (no editar) ---
import os, sys, random, subprocess, importlib, hashlib, json
SESSION = "S01"
FAST_DEV_RUN = os.environ.get("COURSE_FAST_DEV_RUN") == "1"
IN_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ
IN_COLAB = "google.colab" in sys.modules
if FAST_DEV_RUN and not STUDENT_ID.strip():
    STUDENT_ID, STUDENT_NAME = "000000", "Prueba automática"
assert STUDENT_ID.strip(), "⚠️ Escribe tu STUDENT_ID en la primera línea de esta celda y vuelve a ejecutarla."

def ensure(package, import_name=None):
    """Instala un paquete solo si falta."""
    try:
        importlib.import_module(import_name or package)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)

PERSONAL_SEED = int(hashlib.sha256(f"{SESSION}:{STUDENT_ID.strip()}".encode()).hexdigest()[:8], 16)
rng = random.Random(PERSONAL_SEED)

def pick(options):
    """Elige una opción de forma determinística a partir de tu código estudiantil."""
    return rng.choice(list(options))

import numpy as np
import torch
random.seed(PERSONAL_SEED); np.random.seed(PERSONAL_SEED % (2**32)); torch.manual_seed(PERSONAL_SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Sesión {SESSION} | Estudiante: {STUDENT_NAME} ({STUDENT_ID}) | Semilla personal: {PERSONAL_SEED} | Dispositivo: {DEVICE}")
# --- fin de la configuración ---

In [ ]:
# --- Tu configuración personal (no editar) ---
PROMPTS_POOL = [
    "Write a short welcome message for new customers of a digital bank.",
    "Suggest a slogan for a coffee brand from the Andes.",
    "Write a two-sentence apology to a customer whose package arrived late.",
    "Describe the benefits of a mobile data plan for small businesses.",
    "Write a short product description for a reusable water bottle.",
    "Give three tips to reduce customer churn in a telecom company.",
    "Write a friendly reminder about an unpaid invoice.",
    "Explain in two sentences why employees should use strong passwords.",
    "Write a short job ad for a junior data analyst.",
]
TEMPERATURES_POOL = [0.2, 0.7, 1.0, 1.3]
SCENARIOS_POOL = [  # Todas las empresas son ficticias.
    {"empresa": "Andes Bank", "sector": "Banca",
     "necesidad": "Resumir las quejas que llegan al call center para que el área de calidad las priorice.",
     "datos": "Transcripciones con datos personales y financieros de clientes; ~3.000 quejas ya clasificadas por analistas.",
     "volumen_mensual": 40_000, "tokens_entrada_por_solicitud": 900,
     "restricciones": "La política interna prohíbe enviar datos de clientes a terceros; aplica la Ley 1581 de 2012."},
    {"empresa": "NovaTel", "sector": "Telecomunicaciones",
     "necesidad": "Asistente que responde preguntas de clientes sobre planes, tarifas y facturación.",
     "datos": "Catálogo de planes y políticas de facturación que cambia cada mes; sin datos sensibles en las preguntas generales.",
     "volumen_mensual": 300_000, "tokens_entrada_por_solicitud": 1_200,
     "restricciones": "Las respuestas sobre precios deben coincidir exactamente con el catálogo vigente."},
    {"empresa": "Clínica Serena", "sector": "Salud",
     "necesidad": "Extraer campos estructurados (diagnóstico, medicamentos, alergias) de notas clínicas llenas de abreviaturas propias.",
     "datos": "Notas clínicas de pacientes; ~2.000 notas ya anotadas por el equipo médico.",
     "volumen_mensual": 5_000, "tokens_entrada_por_solicitud": 700,
     "restricciones": "Datos de salud sensibles: no pueden salir de la infraestructura de la clínica."},
    {"empresa": "Café Alto Montaña", "sector": "Alimentos y retail",
     "necesidad": "Generar textos para redes sociales y campañas de temporada.",
     "datos": "Descripciones de productos y la guía de marca; ningún dato sensible.",
     "volumen_mensual": 150, "tokens_entrada_por_solicitud": 400,
     "restricciones": "Equipo de 4 personas sin perfil técnico; presupuesto muy limitado."},
    {"empresa": "Bufete Ceiba & Asociados", "sector": "Legal",
     "necesidad": "Buscar y resumir cláusulas relevantes en su archivo de contratos para responder consultas de los abogados.",
     "datos": "20.000 contratos propios, confidenciales, que crecen cada semana.",
     "volumen_mensual": 3_000, "tokens_entrada_por_solicitud": 4_000,
     "restricciones": "Secreto profesional: la información de los clientes es confidencial."},
    {"empresa": "Seguros Cordillera Azul", "sector": "Seguros",
     "necesidad": "Redactar la carta de respuesta a cada reclamación con el tono y el formato legal de la compañía.",
     "datos": "~15.000 cartas históricas aprobadas por el área jurídica; datos personales de asegurados.",
     "volumen_mensual": 80_000, "tokens_entrada_por_solicitud": 1_000,
     "restricciones": "Formato legal estricto; el regulador puede auditar cualquier carta."},
    {"empresa": "Colegio Virtual Arrayanes", "sector": "Educación",
     "necesidad": "Dar retroalimentación a ensayos de estudiantes en español siguiendo la rúbrica institucional.",
     "datos": "Rúbrica institucional y ~500 ensayos calificados por docentes; datos de menores de edad.",
     "volumen_mensual": 2_500, "tokens_entrada_por_solicitud": 1_500,
     "restricciones": "Los estudiantes son menores de edad; los docentes deben revisar la retroalimentación."},
    {"empresa": "Alcaldía de Valle Claro", "sector": "Sector público",
     "necesidad": "Asistente que responde preguntas ciudadanas sobre trámites usando la normativa local.",
     "datos": "Normativa y guías de trámites públicas que cambian con frecuencia; algunas preguntas incluyen datos personales.",
     "volumen_mensual": 50_000, "tokens_entrada_por_solicitud": 1_500,
     "restricciones": "Debe citar la norma en la que se basa cada respuesta; transparencia y trazabilidad."},
]

CONFIG = {
    "prompts": rng.sample(PROMPTS_POOL, 3),
    "temperatures": sorted(rng.sample(TEMPERATURES_POOL, 2)),
    "scenario": pick(SCENARIOS_POOL),
}

print("🎯 Tu configuración personal")
print("\nPrompts asignados (en inglés, porque SmolLM2-135M se entrenó sobre todo con texto en inglés):")
for i, p in enumerate(CONFIG["prompts"], start=1):
    print(f"  P{i}. {p}")
print("\nTemperaturas a comparar:", CONFIG["temperatures"])
esc = CONFIG["scenario"]
print(f"\nEscenario de negocio: {esc['empresa']} ({esc['sector']})")
for campo in ("necesidad", "datos", "volumen_mensual", "tokens_entrada_por_solicitud", "restricciones"):
    print(f"  - {campo}: {esc[campo]}")

## Tarea 1 — Ejecución: genera y mide la diversidad (Ejecución técnica, 40%)

Vas a usar **SmolLM2-135M-Instruct**, el modelo de chat del lab. Para cada uno de tus **3 prompts** y cada una de tus
**2 temperaturas** generarás **5 muestras** (30 textos en total) y medirás:

- **distinct-1** y **distinct-2** (Li et al., 2016): proporción de palabras (1-gramas) y de pares de palabras (2-gramas)
  **únicos** sobre el total en las 5 muestras de una combinación. 1.0 = nada se repite; cerca de 0 = las muestras dicen casi lo mismo.
- **longitud promedio**: número promedio de palabras por muestra.

Para aislar el efecto de la temperatura, el muestreo usa `top_k=0` y `top_p=1.0` (sin filtros): solo cambia la temperatura.
La semilla depende de tu código y del prompt, así que puedes volver a ejecutar y obtener los mismos textos.

**Qué debes hacer:**
1. Ejecuta la celda de apoyo (carga el modelo y define las funciones; no la edites).
2. En la celda ✏️, revisa los parámetros (`N_MUESTRAS = 5`, `MAX_NEW_TOKENS = 48`) y ejecuta el experimento.
3. Revisa la tabla de resultados y lee **todas** las muestras (las necesitas en la Tarea 2).

In [ ]:
# --- Modelo y funciones de apoyo (no editar) ---
ensure("transformers")
import re
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "HuggingFaceTB/SmolLM2-135M-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).float().to(DEVICE)  # float32 en todas las versiones
model.eval()


def codificar_chat(prompt):
    """Aplica la plantilla de chat y devuelve las entradas del modelo (tensores en DEVICE)."""
    enc = tokenizer.apply_chat_template([{"role": "user", "content": prompt}], add_generation_prompt=True,
                                        return_tensors="pt", return_dict=True)
    if not hasattr(enc, "keys"):  # algunas versiones antiguas devuelven un tensor
        enc = {"input_ids": enc, "attention_mask": torch.ones_like(enc)}
    return {k: v.to(DEVICE) for k, v in enc.items()}


def generar_muestras(prompt, temperatura, n, max_new_tokens, semilla):
    """Genera n respuestas muestreadas con una temperatura dada (sin top-k ni top-p)."""
    entradas = codificar_chat(prompt)
    torch.manual_seed(semilla)
    with torch.no_grad():
        salida = model.generate(**entradas, do_sample=True, temperature=temperatura, top_k=0, top_p=1.0,
                                max_new_tokens=max_new_tokens, num_return_sequences=n,
                                pad_token_id=tokenizer.eos_token_id)
    n_entrada = entradas["input_ids"].shape[1]
    return [tokenizer.decode(s[n_entrada:], skip_special_tokens=True).strip() for s in salida]


def palabras(texto):
    """Palabras en minúscula (sin signos de puntuación)."""
    return re.findall(r"\w+(?:'\w+)*", texto.lower())


def distinct_n(textos, n):
    """Proporción de n-gramas únicos sobre el total de n-gramas en un grupo de textos (Li et al., 2016)."""
    ngramas = []
    for texto in textos:
        ws = palabras(texto)
        ngramas += [tuple(ws[i:i + n]) for i in range(len(ws) - n + 1)]
    return len(set(ngramas)) / len(ngramas) if ngramas else 0.0


def metricas_diversidad(textos):
    """distinct-1, distinct-2 y longitud promedio (en palabras) de un grupo de muestras."""
    return {"distinct_1": round(distinct_n(textos, 1), 4),
            "distinct_2": round(distinct_n(textos, 2), 4),
            "longitud_promedio": round(sum(len(palabras(t)) for t in textos) / max(1, len(textos)), 4)}


print(f"Modelo listo: {MODEL_NAME} en {DEVICE}")
print("Prueba de la métrica:", metricas_diversidad(["the bank is open", "the bank is closed today"]))

In [ ]:
# ✏️ TU CÓDIGO AQUÍ — parámetros del experimento (los valores por defecto ya funcionan)
N_MUESTRAS = 5          # muestras por combinación prompt × temperatura
MAX_NEW_TOKENS = 48     # longitud máxima de cada respuesta, en tokens
if FAST_DEV_RUN:        # solo para la prueba automática del curso: no lo cambies
    N_MUESTRAS, MAX_NEW_TOKENS = 2, 12

muestras = {}      # (prompt, temperatura) -> lista de textos
filas = []         # una fila de métricas por combinación
for i, prompt in enumerate(CONFIG["prompts"], start=1):
    for t in CONFIG["temperatures"]:
        textos = generar_muestras(prompt, t, N_MUESTRAS, MAX_NEW_TOKENS, semilla=PERSONAL_SEED + i)
        muestras[(f"P{i}", t)] = textos
        filas.append({"prompt": f"P{i}", "temperatura": t, **metricas_diversidad(textos)})

tabla = pd.DataFrame(filas)
tabla

In [ ]:
# Lee todas las muestras antes de la Tarea 2 (no editar)
for (p, t), textos in muestras.items():
    print(f"\n=== {p} · T = {t} ===")
    for k, texto in enumerate(textos, start=1):
        print(f"  [{k}] {texto!r}")

## Tarea 2 — Análisis: temperatura, diversidad y calidad (Análisis e interpretación, 40%)

La celda siguiente resume tus métricas por temperatura y las grafica. Después:
1. **Califica la calidad** de cada combinación (fila de `tabla`) de 1 a 5 después de leer sus muestras:
   1 = inservible (incoherente o fuera de tema) · 3 = usable con edición · 5 = lista para enviar a un cliente.
2. **Elige la mejor temperatura** para cada prompt en `MEJOR_TEMPERATURA`.
3. Responde en la celda **Tu respuesta:** usando tus números:
   - a) ¿Cómo cambiaron distinct-1, distinct-2 y la longitud al subir la temperatura? Cita los valores.
   - b) ¿Más diversidad significó mejor calidad? Da un ejemplo concreto: cita (entre comillas) una muestra.
   - c) ¿Qué temperatura elegiste para cada prompt y por qué? ¿Depende de si la tarea es creativa o informativa?
   - d) ¿Qué **no** mide distinct-n? Nombra una limitación de esta métrica.

In [ ]:
# Resumen por temperatura (no editar)
import matplotlib.pyplot as plt

resumen = tabla.groupby("temperatura")[["distinct_1", "distinct_2", "longitud_promedio"]].mean().round(4)
display(resumen)

fig, ax = plt.subplots(figsize=(7, 3.5))
resumen[["distinct_1", "distinct_2"]].plot.bar(ax=ax, rot=0)
ax.set(title="Diversidad promedio por temperatura", xlabel="temperatura", ylabel="proporción de n-gramas únicos", ylim=(0, 1.05))
plt.tight_layout()
plt.show()

In [ ]:
# ✏️ TU CÓDIGO AQUÍ — tu lectura de las muestras
print(tabla[["prompt", "temperatura"]].to_string())   # el orden de las filas para CALIDAD

# Una calificación de 1 a 5 por fila de la tabla, en el mismo orden. Ejemplo: [4, 2, 5, 3, 4, 1]
CALIDAD = [None] * len(tabla)

# La mejor temperatura para cada prompt (debe ser una de tus dos temperaturas). Ejemplo: {"P1": 0.7, ...}
MEJOR_TEMPERATURA = {"P1": None, "P2": None, "P3": None}

# --- verificación (no editar) ---
assert len(CALIDAD) == len(tabla), f"CALIDAD debe tener {len(tabla)} valores, uno por fila de la tabla."
tabla["calidad"] = CALIDAD
if any(c is None for c in CALIDAD) or any(v is None for v in MEJOR_TEMPERATURA.values()):
    print("⚠️ Aún faltan calificaciones o temperaturas por elegir.")
tabla

**Tu respuesta:**

a)

b)

c)

d)

## Tarea 3 — Explica y decide: ¿dónde ubicas tu escenario? (Análisis e interpretación, 40%)

Tu escenario de negocio está en `CONFIG["scenario"]` (impreso arriba). Un comité te pide una recomendación:
**¿dónde debe ubicarse en el espectro de adaptación: prompting, RAG, fine-tuning o entrenamiento desde cero?**
¿Y dónde debería correr el modelo: API de un proveedor o modelo abierto en infraestructura propia?

La celda siguiente estima el **volumen mensual de tokens** de tu escenario con tus propios números: tokens de entrada
que da el escenario más los **tokens de salida que midió tu experimento** (con tu mejor temperatura si ya la elegiste).
Si quieres convertirlos en dinero, busca el precio actual de un proveedor y **cita la fuente y la fecha**; no inventes precios.

**Tu respuesta debe incluir:** la posición recomendada; argumentos de **costo, datos, privacidad y control**;
**al menos un riesgo** con su mitigación; un primer paso concreto (piloto); y **al menos un número de tu notebook**.

In [ ]:
# Estimación de tokens con tus números (no editar)
def tokens_salida(textos):
    return [len(tokenizer(t)["input_ids"]) for t in textos]

elegidas = [(p, t) for (p, t) in muestras if MEJOR_TEMPERATURA.get(p) == t]
base = elegidas if elegidas else list(muestras)            # si no has elegido, usa todas las muestras
salidas = [n for clave in base for n in tokens_salida(muestras[clave])]
tokens_salida_promedio = sum(salidas) / max(1, len(salidas))
tokens_por_solicitud = esc["tokens_entrada_por_solicitud"] + tokens_salida_promedio
tokens_mensuales = esc["volumen_mensual"] * tokens_por_solicitud

print(f"Escenario: {esc['empresa']} · {esc['volumen_mensual']:,} solicitudes/mes")
print(f"Tokens de salida promedio medidos en tu experimento: {tokens_salida_promedio:.1f}"
      + ("" if elegidas else " (todas las muestras: aún no eliges temperaturas)"))
print(f"Tokens por solicitud (entrada + salida): {tokens_por_solicitud:,.1f}")
print(f"Tokens estimados por mes: {tokens_mensuales:,.0f} (≈ {tokens_mensuales / 1e6:,.2f} millones)")

In [ ]:
# ✏️ TU DECISIÓN
POSICION_RECOMENDADA = ""   # una de: "prompting", "RAG", "fine-tuning", "desde cero"
DESPLIEGUE = ""             # una de: "API de un proveedor", "modelo abierto en infraestructura propia", "mixto"

OPCIONES_POSICION = {"prompting", "RAG", "fine-tuning", "desde cero"}
OPCIONES_DESPLIEGUE = {"API de un proveedor", "modelo abierto en infraestructura propia", "mixto"}
if POSICION_RECOMENDADA not in OPCIONES_POSICION or DESPLIEGUE not in OPCIONES_DESPLIEGUE:
    print("⚠️ Elige POSICION_RECOMENDADA y DESPLIEGUE entre las opciones indicadas.")
else:
    print(f"Recomendación registrada: {POSICION_RECOMENDADA} · {DESPLIEGUE}")

**Tu respuesta:**

- **Recomendación (posición en el espectro y despliegue):**
- **Costo** (usa tu estimación de tokens):
- **Datos** (qué datos hay y cuáles harían falta):
- **Privacidad:**
- **Control** (versiones, actualizaciones, dependencia del proveedor):
- **Riesgo principal y cómo lo mitigarías:**
- **Primer paso (piloto de 4–6 semanas):**
- **Evidencia de tu notebook que apoya la decisión** (por ejemplo, lo que observaste en la Tarea 2):

## Tarea 4 — Crítica a la IA: BERT vs. GPT (Análisis e interpretación, 40%)

Un colega le pidió a un asistente de IA: *"Explícame en un párrafo en qué se diferencian BERT y GPT"*. Esta fue la respuesta:

> BERT y GPT son modelos basados en la arquitectura Transformer (Vaswani et al., 2017), pero se preentrenan de forma
> distinta. BERT es un modelo autorregresivo: genera texto de izquierda a derecha, un token a la vez, y por eso es la
> opción natural para redactar correos o resúmenes largos. GPT, en cambio, se entrena con modelado de lenguaje
> enmascarado: oculta palabras al azar dentro de la oración y aprende a adivinarlas mirando el contexto de ambos lados.
> Los dos se pueden adaptar a tareas específicas de una empresa, aunque hay que tener en cuenta que hacer fine-tuning
> siempre requiere entrenar el modelo desde cero con los datos de la empresa, lo que lo hace muy costoso. Por último,
> como ambos aprenden de grandes cantidades de texto sin etiquetar y luego se adaptan a muchas tareas, se consideran
> modelos fundacionales (Bommasani et al., 2021).

El texto tiene **3 errores conceptuales**. Para cada uno:
1. Cita la frase incorrecta.
2. Explica por qué es incorrecta.
3. Escribe la corrección.
4. Indica cómo lo verificaste: una fuente primaria, las slides o una evidencia del lab (puedes usar la celda de evidencia).

Cierra con el párrafo corregido completo.

In [ ]:
# ✏️ (Opcional) Evidencia: ¿qué hace BERT con un [MASK]? Cambia la frase si quieres probar otra idea.
from transformers import pipeline

fill_mask = pipeline("fill-mask", model="google-bert/bert-base-uncased", device=DEVICE)
frase = "The bank will [MASK] your new card tomorrow morning."
for pred in fill_mask(frase, top_k=5):
    print(f"{pred['score']:.3f}  {pred['token_str']}")

**Tu respuesta:**

| # | Frase incorrecta | Por qué es incorrecta | Corrección | Cómo lo verificaste |
|---|---|---|---|---|
| 1 |  |  |  |  |
| 2 |  |  |  |  |
| 3 |  |  |  |  |

**Párrafo corregido:**

In [ ]:
# --- Resultados clave (no editar: se llenan con tus variables) ---
RESULTS = {
    "metricas_por_temperatura": {
        f"T={t}": {col: round(float(v), 4) for col, v in fila.items()}
        for t, fila in resumen.iterrows()
    },
    "metricas_por_prompt": [
        {"prompt": f["prompt"], "temperatura": float(f["temperatura"]),
         "distinct_1": round(float(f["distinct_1"]), 4), "distinct_2": round(float(f["distinct_2"]), 4),
         "longitud_promedio": round(float(f["longitud_promedio"]), 4), "calidad": f["calidad"]}
        for f in tabla.to_dict("records")
    ],
    "mejor_temperatura_por_prompt": MEJOR_TEMPERATURA,
    "tokens_salida_promedio": round(float(tokens_salida_promedio), 4),
    "tokens_mensuales_estimados": round(float(tokens_mensuales), 4),
    "posicion_recomendada": POSICION_RECOMENDADA,
    "despliegue": DESPLIEGUE,
    "n_muestras": N_MUESTRAS,
    "max_new_tokens": MAX_NEW_TOKENS,
}
print("✅ RESULTS listo. Ejecuta la última celda para ver tu huella de resultados.")

## 🪞 Reflexión y registro de uso de IA (20%)

### Reflexión (responde en 3–5 líneas cada una)
1. ¿Qué aprendiste hoy que puedas aplicar en tu trabajo u organización?
2. ¿Qué fue lo más difícil y cómo lo resolviste?
3. Autoevaluación: ¿qué nota (0.0–5.0) te pondrías en este challenge y por qué?

### 🤖 Registro de uso de IA (obligatorio)
| # | Herramienta (ChatGPT, Claude, Codex, Copilot…) | ¿Para qué la usaste? | Prompt principal (resumido) | ¿Qué verificaste o corregiste de su respuesta? |
|---|---|---|---|---|
| 1 |  |  |  |  |
| 2 |  |  |  |  |

> Si no usaste asistentes de IA, escribe: *"No usé asistentes de IA en este challenge."* Recuerda: puedes usar IA, pero **eres responsable de todo lo que entregas** y podrías ser seleccionado(a) para una micro-sustentación.

In [ ]:
# --- Huella de resultados (no editar) ---
def results_fingerprint(results):
    payload = json.dumps({"session": SESSION, "student": STUDENT_ID.strip(), "results": results},
                         sort_keys=True, default=str, ensure_ascii=False)
    return hashlib.sha256(payload.encode()).hexdigest()[:12]

print(json.dumps({"CONFIG": CONFIG, "RESULTS": RESULTS}, indent=2, ensure_ascii=False, default=str))
print("🔏 Huella de resultados:", results_fingerprint({"CONFIG": CONFIG, "RESULTS": RESULTS}))